# 🎬 Faceless Review Video Generator — Colab Pipeline

This notebook runs the full pipeline:
1. **Script Generation** — OpenRouter API (DeepSeek, Llama, etc.)
2. **Audio Generation** — Edge-TTS (Microsoft free TTS)
3. **Image Generation** — Flux.1-schnell via Diffusers (local on T4 GPU)
4. **Video Rendering** — Wan 2.1 or LTX-Video (selectable)
5. **Composition** — MoviePy + FFmpeg stitch everything together

---
**⚠️ Runtime**: Go to `Runtime` → `Change runtime type` → Select `T4 GPU`

## 1. Install Dependencies

In [ ]:
# Install all required packages
!pip install -q gradio>=4.0.0 requests>=2.31.0 diffusers>=0.27.0 transformers>=4.40.0
!pip install -q accelerate>=0.28.0 torch>=2.1.0 torchvision>=0.16.0
!pip install -q edge-tts>=6.1.0 moviepy>=1.0.3 ffmpeg-python>=0.2.0
!pip install -q imageio-ffmpeg>=0.4.9 pillow>=10.0.0 numpy>=1.24.0
!pip install -q openai>=1.0.0

print('✅ All dependencies installed!')

## 2. Clone Repository (if not already cloned)

In [ ]:
import os
import sys

# If running from Colab, you can clone the repo:
REPO_URL = "https://github.com/Motchucuncon/AllInOne.git"
REPO_DIR = "/content/AllInOne"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print(f'✅ Repository cloned to {REPO_DIR}')
else:
    print(f'✅ Repository already exists at {REPO_DIR}')

# Change to repo directory
%cd {REPO_DIR}

# Add to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('✅ Working directory:', os.getcwd())

## 3. Set OpenRouter API Key

Get your API key from [OpenRouter.ai](https://openrouter.ai/keys) and paste it below.

In [ ]:
import os
from getpass import getpass

# Set your OpenRouter API key
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print('✅ API key set!')

## 4. Configure Pipeline Parameters

Edit the parameters below to customize your video.

In [ ]:
# ============================================================
# CONFIGURATION — Edit these values
# ============================================================

# Review topic (Vietnamese)
TOPIC = "Đánh giá iPhone 15 Pro Max"

# OpenRouter model
OPENROUTER_MODEL = "deepseek/deepseek-r1:free"

# Video model: "wan_2_1" or "ltx_video"
VIDEO_MODEL = "wan_2_1"

# TTS voice: "vi-VN-HoaiMyNeural" or "vi-VN-NamMinhNeural"
TTS_VOICE = "vi-VN-HoaiMyNeural"

# Output directory
OUTPUT_DIR = "/content/output"

print(f'✅ Configuration loaded!')
print(f'   Topic: {TOPIC}')
print(f'   OpenRouter model: {OPENROUTER_MODEL}')
print(f'   Video model: {VIDEO_MODEL}')
print(f'   TTS voice: {TTS_VOICE}')

## 5. Run Pipeline - Step by Step

### 5.1 Generate Storyboard

In [ ]:
import json
from core.script_gen import generate_storyboard

print('🤖 Generating storyboard with OpenRouter...')

storyboard = generate_storyboard(
    topic=TOPIC,
    model=OPENROUTER_MODEL,
    api_key=OPENROUTER_API_KEY,
)

# Save storyboard
os.makedirs(OUTPUT_DIR, exist_ok=True)
storyboard_path = os.path.join(OUTPUT_DIR, "storyboard.json")
with open(storyboard_path, "w", encoding="utf-8") as f:
    json.dump(storyboard, f, ensure_ascii=False, indent=2)

print(f'✅ Storyboard generated with {len(storyboard["storyboard_scenes"])} scenes!')
print(f'\n📝 Narration preview:')
print(storyboard['full_narration'][:300] + '...')
print(f'\n📋 Storyboard saved to: {storyboard_path}')

### 5.2 Generate Audio & Subtitles

In [ ]:
from core.audio_gen import generate_audio_and_subtitles

print('🔊 Generating narration audio with Edge-TTS...')

audio_result = generate_audio_and_subtitles(
    storyboard=storyboard,
    voice=TTS_VOICE,
    output_dir=OUTPUT_DIR,
)

print(f'✅ Audio generated: {audio_result["audio_path"]}')
print(f'✅ Subtitles generated: {audio_result["subtitles_path"]}')
print(f'⏱️ Duration: {audio_result["duration_seconds"]:.1f}s')

### 5.3 Generate B-roll Images (Flux.1-schnell)

In [ ]:
from core.image_gen import generate_broll_images

print('🖼️ Generating B-roll images with Flux.1-schnell...')

scenes = storyboard['storyboard_scenes']
broll_prompts = [s['broll_prompt'] for s in scenes]
scene_ids = [s['scene_id'] for s in scenes]

image_paths = generate_broll_images(
    prompts=broll_prompts,
    scene_ids=scene_ids,
    output_dir=os.path.join(OUTPUT_DIR, "images"),
    unload_after=True,
)

print(f'✅ Generated {len(image_paths)} B-roll images!')

### 5.4 Render Video Clips

In [ ]:
from core.video_gen import render_video_clips

print(f'🎬 Rendering video clips with {VIDEO_MODEL}...')

motion_prompts = [s.get('broll_prompt', '') for s in scenes]

clip_paths = render_video_clips(
    image_paths=image_paths,
    prompts=motion_prompts,
    scene_ids=scene_ids,
    model=VIDEO_MODEL,
    output_dir=os.path.join(OUTPUT_DIR, "clips"),
)

print(f'✅ Rendered {len(clip_paths)} video clips!')

### 5.5 Compose Final Video

In [ ]:
from core.composer import compose_final_video

print('🎞️ Composing final video...')

final_video_path = compose_final_video(
    clip_paths=clip_paths,
    audio_path=audio_result['audio_path'],
    subtitles_path=audio_result['subtitles_path'],
    output_dir=OUTPUT_DIR,
)

print(f'✅ Final video created: {final_video_path}')

## 6. Download Results

Download the generated files to your local machine.

In [ ]:
from google.colab import files
from IPython.display import display, Video, HTML

# Display the final video
if os.path.exists(final_video_path):
    display(Video(final_video_path, width=640))
    print(f'\n📁 Final video: {final_video_path}')

# Download options
print('\n📥 Click the button below to download the final video:')
files.download(final_video_path)

print('\n📥 Or download all outputs as a zip:')
!zip -r /content/output.zip {OUTPUT_DIR}
files.download('/content/output.zip')

## 7. Launch Gradio Web UI (Optional)

Run the interactive Gradio interface instead of the step-by-step pipeline.

In [ ]:
# Launch the Gradio web interface
print('🚀 Launching Gradio UI...')

import logging
logging.getLogger('gradio').setLevel(logging.ERROR)

# Run app.py with share=True
!python app.py &

import time
time.sleep(5)

from IPython.display import display, HTML
display(HTML("<p>✅ Gradio app launched! Use the Gradio share URL from the output above.</p>"))

---

## Pipeline Complete 🎉

Your faceless review video has been generated! Check the output folder for:
- `output/final_review_video.mp4` — The final video
- `output/narration.mp3` — Narration audio
- `output/subtitles.srt` — Subtitle file
- `output/storyboard.json` — Generated storyboard
- `output/images/` — B-roll keyframe images
- `output/clips/` — Individual video clips